In [3]:
import pyrtl
import numpy as np

In [7]:
def create_matrix_multiply(SIZE=16, TILE_SIZE=2):
    """
    Create a matrix multiplication circuit with corrected memory writes
    """
    # Reset any working blocks
    pyrtl.reset_working_block()

    # Calculate required bits for addressing
    ADDR_BITS = (SIZE * SIZE).bit_length()
    DATA_BITS = 16  # Using 16-bit fixed point numbers
    
    # Control inputs
    start = pyrtl.Input(1, 'start')
    reset = pyrtl.Input(1, 'reset')
    
    # Memory interface
    rd_addr_a = pyrtl.Register(ADDR_BITS, 'rd_addr_a')
    rd_addr_b = pyrtl.Register(ADDR_BITS, 'rd_addr_b')
    wr_addr_c = pyrtl.Register(ADDR_BITS, 'wr_addr_c')
    wr_en_c = pyrtl.WireVector(1, 'wr_en_c')  # Changed to WireVector
    
    # Status outputs
    done = pyrtl.Output(1, 'done')
    busy = pyrtl.Output(1, 'busy')
    
    # Debug outputs
    debug_i = pyrtl.Output(ADDR_BITS, 'debug_i')
    debug_j = pyrtl.Output(ADDR_BITS, 'debug_j')
    debug_k = pyrtl.Output(ADDR_BITS, 'debug_k')
    debug_acc = pyrtl.Output(DATA_BITS, 'debug_acc')
    
    # Create memories
    mem_a = pyrtl.MemBlock(bitwidth=DATA_BITS, addrwidth=ADDR_BITS, name='mem_a')
    mem_b = pyrtl.MemBlock(bitwidth=DATA_BITS, addrwidth=ADDR_BITS, name='mem_b')
    mem_c = pyrtl.MemBlock(bitwidth=DATA_BITS, addrwidth=ADDR_BITS, name='mem_c')
    
    # Create registers
    a_reg = pyrtl.Register(DATA_BITS, 'a_reg')
    b_reg = pyrtl.Register(DATA_BITS, 'b_reg')
    
    # State machine registers
    state = pyrtl.Register(3, 'state')
    i = pyrtl.Register(ADDR_BITS, 'i')
    j = pyrtl.Register(ADDR_BITS, 'j')
    k = pyrtl.Register(ADDR_BITS, 'k')
    
    # Accumulator
    accum = pyrtl.Register(DATA_BITS, 'accum')
    
    # Debug connections
    debug_i <<= i
    debug_j <<= j
    debug_k <<= k
    debug_acc <<= accum
    
    # State definitions
    IDLE = pyrtl.Const(0, 3)
    PREP_ADDR = pyrtl.Const(1, 3)
    WAIT_DATA = pyrtl.Const(2, 3)
    COMPUTE = pyrtl.Const(3, 3)
    WRITE_RESULT = pyrtl.Const(4, 3)
    
    # Write enable logic
    wr_en_c <<= (state == WRITE_RESULT)
    
    # State machine logic
    with pyrtl.conditional_assignment:
        with reset:
            state.next |= IDLE
            i.next |= 0
            j.next |= 0
            k.next |= 0
            rd_addr_a.next |= 0
            rd_addr_b.next |= 0
            wr_addr_c.next |= 0
            accum.next |= 0
            done |= 0
            busy |= 0
            
        with (state == IDLE) & start:
            state.next |= PREP_ADDR
            busy |= 1
            done |= 0
            
        with state == PREP_ADDR:
            rd_addr_a.next |= (i * SIZE) + k
            rd_addr_b.next |= (k * SIZE) + j
            state.next |= WAIT_DATA
            
        with state == WAIT_DATA:
            a_reg.next |= mem_a[rd_addr_a]
            b_reg.next |= mem_b[rd_addr_b]
            state.next |= COMPUTE
            
        with state == COMPUTE:
            # Perform multiplication and accumulation
            product = pyrtl.concat_list([a_reg, b_reg])  # Ensure proper bit width
            accum.next |= accum + product[:DATA_BITS]  # Take appropriate bits
            
            with k < SIZE - 1:
                k.next |= k + 1
                state.next |= PREP_ADDR
            with k == SIZE - 1:
                k.next |= 0
                wr_addr_c.next |= (i * SIZE) + j
                state.next |= WRITE_RESULT
            
        with state == WRITE_RESULT:
            with j < SIZE - 1:
                j.next |= j + 1
                state.next |= PREP_ADDR
                accum.next |= 0
            with j == SIZE - 1:
                j.next |= 0
                with i < SIZE - 1:
                    i.next |= i + 1
                    state.next |= PREP_ADDR
                    accum.next |= 0
                with i == SIZE - 1:
                    state.next |= IDLE
                    done |= 1
                    busy |= 0
                    accum.next |= 0

    # Write result to memory C
    mem_c[wr_addr_c] <<= pyrtl.select(wr_en_c, accum, mem_c[wr_addr_c])

    return mem_a, mem_b, mem_c, start, reset, done, busy

In [5]:
# Test function
def test_matrix_multiply():
    # Create 8x8 matrix multiplier
    mem_a, mem_b, mem_c, start, reset, done, busy = create_matrix_multiply(SIZE=8)
    
    # Initialize test matrices (8x8)
    matrix_a = {i: i + 1 for i in range(64)}  # Values 1-8 repeated in rows
    matrix_b = {i: 1 for i in range(64)}         # All ones
    
    # Create simulation
    sim = pyrtl.Simulation(memory_value_map={
        mem_a: matrix_a,
        mem_b: matrix_b
    })
    
    # Reset circuit
    sim.step({
        'reset': 1,
        'start': 0
    })
    
    # Start multiplication
    sim.step({
        'reset': 0,
        'start': 1
    })
    
    # Run until done
    cycles = 0
    while not sim.inspect('done') and cycles < 2000:  # Added cycle limit
        sim.step({
            'reset': 0,
            'start': 0
        })
        cycles += 1
        
        # Debug prints every 100 cycles
        if cycles % 100 == 0:
            print(f"Cycle {cycles}:")
            print(f"State: {sim.inspect('state')}")
            print(f"i,j,k: {sim.inspect('i')},{sim.inspect('j')},{sim.inspect('k')}")
            print(f"Accumulator: {sim.inspect('accum')}")
            print(f"Write Enable: {sim.inspect('wr_en_c')}")
            print("---")
    
    print(f"\nMatrix multiplication completed in {cycles} cycles!")
    
    # Print results
    mem_c_data = sim.inspect_mem(mem_c)
    print("\nOutput matrix C:")
    for i in range(8):
        row = [str(mem_c_data.get(i*8 + j, 0)).rjust(4) for j in range(8)]
        print(f"Row {i}: {' '.join(row)}")

In [6]:
test_matrix_multiply()

Cycle 100:
State: 4
i,j,k: 0,3,0
Accumulator: 36
Write Enable: 1
---
Cycle 200:
State: 4
i,j,k: 0,7,0
Accumulator: 36
Write Enable: 1
---
Cycle 300:
State: 4
i,j,k: 1,3,0
Accumulator: 100
Write Enable: 1
---
Cycle 400:
State: 4
i,j,k: 1,7,0
Accumulator: 100
Write Enable: 1
---
Cycle 500:
State: 4
i,j,k: 2,3,0
Accumulator: 164
Write Enable: 1
---
Cycle 600:
State: 4
i,j,k: 2,7,0
Accumulator: 164
Write Enable: 1
---
Cycle 700:
State: 4
i,j,k: 3,3,0
Accumulator: 228
Write Enable: 1
---
Cycle 800:
State: 4
i,j,k: 3,7,0
Accumulator: 228
Write Enable: 1
---
Cycle 900:
State: 4
i,j,k: 4,3,0
Accumulator: 292
Write Enable: 1
---
Cycle 1000:
State: 4
i,j,k: 4,7,0
Accumulator: 292
Write Enable: 1
---
Cycle 1100:
State: 4
i,j,k: 5,3,0
Accumulator: 356
Write Enable: 1
---
Cycle 1200:
State: 4
i,j,k: 5,7,0
Accumulator: 356
Write Enable: 1
---
Cycle 1300:
State: 4
i,j,k: 6,3,0
Accumulator: 420
Write Enable: 1
---
Cycle 1400:
State: 4
i,j,k: 6,7,0
Accumulator: 420
Write Enable: 1
---
Cycle 1500:
State